# Interactive Visualization Lab

Complete the following set of exercises to solidify your knowledge of interactive visualization using Plotly, Cufflinks, and IPyWidgets.

In [1]:
import pandas as pd
#import plotly.plotly as py
import plotly.express as px
import cufflinks as cf
from ipywidgets import interact

cf.go_offline()

In [2]:
data = pd.read_excel(r'G:\My Drive\Ironhack\SQL\lab-interactive-visualization\data\Online Retail.xlsx')

In [3]:
data.head()

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,Revenue,CustomerID,Country
0,536365,2010-12-01 08:26:00,85123A,CREAM HANGING HEART T-LIGHT HOLDER,6,2.55,15.3,17850,United Kingdom
1,536373,2010-12-01 09:02:00,85123A,CREAM HANGING HEART T-LIGHT HOLDER,6,2.55,15.3,17850,United Kingdom
2,536375,2010-12-01 09:32:00,85123A,CREAM HANGING HEART T-LIGHT HOLDER,6,2.55,15.3,17850,United Kingdom
3,536390,2010-12-01 10:19:00,85123A,CREAM HANGING HEART T-LIGHT HOLDER,64,2.55,163.2,17511,United Kingdom
4,536394,2010-12-01 10:39:00,85123A,CREAM HANGING HEART T-LIGHT HOLDER,32,2.55,81.6,13408,United Kingdom


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396034 entries, 0 to 396033
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    396034 non-null  int64         
 1   InvoiceDate  396034 non-null  datetime64[ns]
 2   StockCode    396034 non-null  object        
 3   Description  396034 non-null  object        
 4   Quantity     396034 non-null  int64         
 5   UnitPrice    396034 non-null  float64       
 6   Revenue      396034 non-null  float64       
 7   CustomerID   396034 non-null  int64         
 8   Country      396034 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 27.2+ MB


In [5]:
data.drop_duplicates(inplace=True)

## 1. Create an interactive bar chart showing total quantity and revenue by country (excluding United Kingdom) for the month of April 2011.

In [6]:
data_no_uk_apr_2011 = data.loc[(data["Country"] != "United Kingdom") & (data["InvoiceDate"].dt.month == 4) & (data["InvoiceDate"].dt.year == 2011)]


In [7]:
def plot_bar_chart():
    fig = px.bar(data_no_uk_apr_2011, 
                 x="Country", 
                 y=["Revenue", "Quantity"], 
                 barmode="group", 
                 title="Total Quantity and Revenue by Country (April 2011, Excluding UK)")
    fig.show()
plot_bar_chart()

## 2. Create an interactive line chart showing quantity and revenue sold to France between January 1st and May 31st 2011.

In [8]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Create list of unique sorted dates for the slider
date_range = sorted(data["InvoiceDate"].dt.date.unique())

# Date range slider
date_slider = widgets.SelectionRangeSlider(
    options=date_range,
    index=(0, len(date_range)-1),
    description='Date Range',
    layout={'width': '95%'},
    style={'description_width': 'initial'}
)

# Aggregation dropdown
aggregation_dropdown = widgets.Dropdown(
    options=["Daily", "Monthly"],
    value="Daily",
    description="Aggregate by:"
)

# Combine widgets
ui = widgets.VBox([date_slider, aggregation_dropdown])

def update_chart(date_range, aggregation):
    start_date, end_date = date_range

    # Filter
    df = data[
        (data["Country"] == "France") & 
        (data["InvoiceDate"] >= pd.to_datetime(start_date)) & 
        (data["InvoiceDate"] <= pd.to_datetime(end_date))
    ][["InvoiceDate", "Quantity", "Revenue"]]

    if df.empty:
        print("No data available for selected range.")
        return

    # Group by period
    if aggregation == "Monthly":
        df["Period"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
    else:
        df["Period"] = df["InvoiceDate"].dt.date

    df_agg = df.groupby("Period")[["Quantity", "Revenue"]].sum().reset_index()
    df_melted = df_agg.melt(id_vars="Period", var_name="Metric", value_name="Value")

    fig = px.line(df_melted, x="Period", y="Value", color="Metric", title="France Performance")
    fig.update_layout(xaxis_title="Date", yaxis_title="Value")
    fig.show()

# Connect widgets
out = widgets.interactive_output(update_chart, {
    'date_range': date_slider,
    'aggregation': aggregation_dropdown
})

display(ui, out)


Output()

## 3. Create an interactive scatter plot showing the relationship between average quantity (x-axis) and average unit price (y-axis) for the product PARTY BUNTING with the plot points color-coded by country (categories).

In [9]:
@interact
def bunting_scatter(country="all", product="PARTY BUNTING"):
    product_data = data[data["Description"] == product][["Quantity", "UnitPrice", "Country"]]
    fig = px.scatter(product_data, x="Quantity", y="UnitPrice", color="Country")
    fig.show()

interactive(children=(Text(value='all', description='country'), Text(value='PARTY BUNTING', description='produ…

## 4. Create a set of interactive histograms showing the distributions of quantity per invoice for the following countries: EIRE, Germany, France, and Netherlands.

In [10]:
hist_data = data[(data["Country"].isin(["EIRE", "Germany", "France", "Netherlands"]))&(data["Quantity"]<500)][["InvoiceNo", "Quantity", "Country"]]

In [11]:
@interact(country=["EIRE", "Germany", "France", "Netherlands"])
def plot_histogram(country):
    country_data = hist_data[hist_data["Country"] == country]
    fig = px.histogram(country_data, x="Quantity", nbins=20, title=f"Quantity Distribution for {country}")
    fig.update_layout(xaxis_title="Quantity", yaxis_title="Count")
    fig.show()

interactive(children=(Dropdown(description='country', options=('EIRE', 'Germany', 'France', 'Netherlands'), va…

## 5. Create an interactive side-by-side bar chart showing the revenue by country listed below (bars) for each of the products listed below.

In [12]:
product_list = ['JUMBO BAG RED RETROSPOT', 
                'CREAM HANGING HEART T-LIGHT HOLDER',
                'REGENCY CAKESTAND 3 TIER']

country_list = ['EIRE', 'Germany', 'France', 'Netherlands']

In [13]:
data_filtered = data[(data["Country"].isin(country_list))& (data["Description"].isin(product_list))]

In [14]:
@interact(product=["Select All"] + product_list)
def plot_side_by_side_bar_chart(product):
    if product == "Select All":
        product_data = data_filtered
        title = "Revenue by Country for All Products"
    else:
        product_data = data_filtered[data_filtered["Description"] == product]
        title = f"Revenue by Country for {product}"
    
    fig = px.bar(product_data, x="Country", y="Revenue", color="Country", barmode="group", title=title)
    fig.update_layout(xaxis_title="Country", yaxis_title="Revenue")
    fig.show()

interactive(children=(Dropdown(description='product', options=('Select All', 'JUMBO BAG RED RETROSPOT', 'CREAM…

## 6. Create an interactive line chart showing quantity sold by day for the United Kingdom. Add drop-down boxes for Year and Month that allow you to filter the date range that appears in the chart.

In [15]:
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px

# Load and prepare your data
data['Year'] = data['InvoiceDate'].dt.year
data['Month'] = data['InvoiceDate'].dt.month
data['Day'] = data['InvoiceDate'].dt.day

uk = data[data['Country'] == 'United Kingdom']

# Initialize Dash app
app = dash.Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("UK Sales: Quantity Sold by Day"),

    html.Label("Select Year:"),
    dcc.Dropdown(
        id='year-dropdown',
        options=[{'label': str(y), 'value': y} for y in sorted(uk['Year'].unique())],
        value=uk['Year'].min()
    ),

    html.Label("Select Month:"),
    dcc.Dropdown(id='month-dropdown'),

    dcc.Graph(id='line-chart')
])

# Update month dropdown based on selected year
@app.callback(
    Output('month-dropdown', 'options'),
    Output('month-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_month_options(selected_year):
    months = uk[uk['Year'] == selected_year]['Month'].unique()
    options = [{'label': f'{m:02d}', 'value': m} for m in sorted(months)]
    return options, options[0]['value'] if options else None

# Update line chart based on year and month
@app.callback(
    Output('line-chart', 'figure'),
    Input('year-dropdown', 'value'),
    Input('month-dropdown', 'value')
)
def update_chart(year, month):
    filtered = uk[(uk['Year'] == year) & (uk['Month'] == month)]
    filtered['Date'] = pd.to_datetime(filtered[['Year', 'Month', 'Day']])
    df_grouped = filtered.groupby('Date')['Quantity'].sum().reset_index()

    fig = px.line(df_grouped, x='Date', y='Quantity',
                  title=f'Quantity Sold in {year}-{month:02d}')
    return fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)


## 7. Create an interactive scatter plot that plots number of invoices (x-axis) vs. number of customers (y-axis) and the plot points represent individual products. Add two sliders that control the x and y axis ranges.

In [16]:
agg_func = {'InvoiceNo':'nunique',
            'Quantity':'sum',
            'UnitPrice':'mean',
            'Revenue':'sum',
            'CustomerID':'nunique'}

products = uk.groupby('Description').agg(agg_func)

In [17]:
products

,InvoiceNo,Quantity,UnitPrice,Revenue,CustomerID
Description,,,,,
4 PURPLE FLOCK DINNER CANDLES,35,132,2.305556,250.36,30
50'S CHRISTMAS GIFT BAG LARGE,100,1721,1.247900,2067.25,98
DOLLY GIRL BEAKER,100,657,1.250000,821.25,77
I LOVE LONDON MINI BACKPACK,55,180,4.150000,747.00,46
NINE DRAWER OFFICE TIDY,25,43,14.754000,613.45,24
...,...,...,...,...,...
ZINC T-LIGHT HOLDER STARS SMALL,220,4258,0.839005,3399.62,168
ZINC TOP 2 DOOR WOODEN SHELF,9,10,16.950000,169.50,9
ZINC WILLIE WINKIE CANDLE STICK,169,2006,0.877209,1716.02,123


In [19]:
# Create list of unique sorted dates for the slider
invoice_range = sorted(products["InvoiceNo"].unique())
customer_range = sorted(products["CustomerID"].unique())

# Date range slider
invoice_slider = widgets.SelectionRangeSlider(
    options=invoice_range,
    index=(0, len(invoice_range)-1),
    description='Invoice Count Range',
    layout={'width': '95%'},
    style={'description_width': 'initial'}
)

customer_slider = widgets.SelectionRangeSlider(
    options=customer_range,
    index=(0, len(customer_range)-1),
    description='Customer Count Range',
    layout={'width': '95%'},
    style={'description_width': 'initial'}
)
# Combine widgets
ui = widgets.VBox([invoice_slider, customer_slider])

def update_chart(invoice_range, customer_range):
    start_inv, end_inv = invoice_range
    start_cust, end_cust = customer_range

    # Filter
    df = products[
        (products["InvoiceNo"] >= start_inv) &
        (products["InvoiceNo"] <= end_inv) &
        (products["CustomerID"] >= start_cust) &
        (products["CustomerID"] <= end_cust)
    ]
    if df.empty:
        print("No data available for selected range.")
        return

    fig = px.scatter(df, x="InvoiceNo", y="CustomerID")
    fig.show()

# Connect widgets
out = widgets.interactive_output(update_chart, {
    'invoice_range': invoice_slider,
    'customer_range': customer_slider
})

display(ui, out)


Output()

## 8. Creat an interactive bar chart that shows revenue by product description. Add a text field widget that filters the results to show the product that contain the text entered in their description.

In [ ]:
from ipywidgets import Text, VBox
from IPython.display import display

# Text field widget for filtering
filter_text = Text(
    value='',
    placeholder='Enter text to filter products',
    description='Filter:',
    layout={'width': '50%'},
    style={'description_width': 'initial'}
)

def update_bar_chart(filter_text):
    # Filter data based on the entered text
    filtered_data = products[products.index.str.contains(filter_text, case=False, na=False)]
    
    if filtered_data.empty:
        print("No products match the filter.")
        return
    
    # Create bar chart
    fig = px.bar(
        filtered_data.reset_index(),
        x='Description',
        y='Revenue',
        title='Revenue by Product Description',
        labels={'Revenue': 'Revenue', 'Description': 'Product Description'}
    )
    fig.update_layout(xaxis_title="Product Description", yaxis_title="Revenue")
    fig.show()

# Connect the widget to the function
out = widgets.interactive_output(update_bar_chart, {'filter_text': filter_text})

# Display the widget and output
display(VBox([filter_text, out]))